<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/02_backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Backtest

Notebook `01` produced a score. This notebook asks whether the score is *worth*
anything: do higher-scoring names earn higher forward returns? The method is an
**event study** — on each rebalance date, score the universe, sort names into
buckets by score, and measure their return over a fixed horizon. Aggregated across
many dates, the result shows whether congressional-buying conviction precedes price
moves, and by how much.

### What the backtest reports

- **bucket returns** — average forward return by score bucket. A signal with edge
  produces a rising staircase from the lowest bucket to the highest.
- **information coefficient (IC)** — the rank correlation between score and forward
  return, averaged across dates. A small positive IC that holds in most periods is
  the hallmark of a real, if modest, signal.
- **long-short spread** — the top bucket's return minus the bottom's, each period.
- **equity curve and risk metrics** — a simple strategy that holds the top bucket,
  with Sharpe, drawdown, and hit rate.
- **feature attribution** — forward returns split by committee alignment, which is
  how a feature earns or loses its weight.


## Setup

We pull the latest repo state, install dependencies including yfinance, and read the
Quiver token from Secrets. The backtest defaults to a long **mock history**; set
`USE_LIVE = True` for real Quiver data.

Committee-sector resolution uses a built-in static map, so the backtest makes no
per-ticker network calls and will not hit yfinance rate limits.


In [1]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests yfinance

import subprocess, os, sys, logging
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # quiet delisted-ticker notices
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True to backtest on live Quiver data

Working in: /content/quant-equity-research


## Install the package

This notebook's code lives in the repository under `src/quant_research/`; the clone above brought it in. The cell below installs the package in editable mode and puts it on the session path.

In [2]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.2.0


## Step 1 — Validate the engine before trusting it

Before backtesting the real signal, we confirm the engine behaves on data where the
answer is known: random-walk prices with one score that genuinely predicts the
next-horizon return and one that is pure noise. A trustworthy engine reports strong
edge for the first and nothing for the second.


In [3]:
import numpy as np, pandas as pd
from quant_research.backtest.engine import event_study

rng = np.random.default_rng(0)
tk = [f"T{i:02d}" for i in range(40)]
dates = pd.bdate_range("2021-01-01", periods=600)
synth = pd.DataFrame(100*np.exp(np.cumsum(rng.normal(0,0.01,(len(dates),len(tk))),0)),
                     index=dates, columns=tk)
rebal = pd.bdate_range("2021-02-01", "2022-12-01", freq="W-FRI"); H = 21

def score_edge(d):
    pos = synth.index.searchsorted(pd.Timestamp(d))
    if pos+H >= len(synth.index): return None
    fwd = synth.iloc[pos+H]/synth.iloc[pos]-1
    return pd.Series(fwd + rng.normal(0,0.02,len(fwd)), index=synth.columns)

for label, fn in [("edge present", score_edge),
                  ("random noise", lambda d: pd.Series(rng.normal(size=len(tk)), index=tk))]:
    r = event_study(fn, synth, rebal, horizon=H)["summary"]
    print(f"{label:14s} | mean IC {r['mean_ic']:+.3f} | long-short {r['mean_long_short']:+.4f} "
          f"| LS positive {r['ls_positive_share']:.0%}")

edge present   | mean IC +0.896 | long-short +0.1109 | LS positive 100%
random noise   | mean IC +0.001 | long-short +0.0014 | LS positive 54%


A high IC and a consistently positive spread for the real signal, and effectively
zero for noise. That null result is the important one: it shows the engine is not
quietly using future information.


## Step 2 — Backtest the congress signal

We pull the trade history once, fetch prices for the names that appear, and score the
universe on weekly rebalance dates. `START_DATE` narrows the window to recent, denser
years; widen it to study the full history.

A note on the mock run: the synthetic trades carry a built-in tilt toward a few
tickers, so any apparent edge there mostly reflects how those names happened to move.
The genuine measurement comes from the live run.


In [4]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal
from quant_research.enrichment.live import LiveEnrichment
from quant_research.backtest import prices as P, engine as E

START_DATE = "2022-01-01"     # widen to study more history
UNIVERSE_CAP = 120            # most-traded names, capped for a tractable price pull
HORIZON = 21                  # ~1 trading month forward

if USE_LIVE:
    client = QuiverClient(token=QUIVER_TOKEN)
    trades = client.congress_trades(historical=True)
else:
    client = QuiverClient(mock=True, mock_history_days=540)
    trades = client.congress_trades()

enrichment = LiveEnrichment()    # static sectors, no per-ticker network calls
signal = CongressSignal(client, lookback_days=30, enrichment=enrichment)

trades = trades[trades.report_date >= pd.Timestamp(START_DATE)]
buys = trades[trades.transaction_type == "Purchase"]
counts = buys.ticker.value_counts()
universe = counts[counts >= 3].index.tolist()[:UNIVERSE_CAP]

start = max(buys.report_date.min().date(), pd.Timestamp(START_DATE).date())
end = buys.report_date.max().date()
print(f"{len(trades)} trades | {len(universe)} tickers | {start} to {end}")

price_panel = P.price_history(universe, start, end)
if price_panel.empty or price_panel.shape[1] == 0:
    print("\n[!] price panel is empty — yfinance may be rate-limiting.")
    print("    Wait a few minutes (or restart the runtime), then re-run this cell.")
else:
    print("price panel:", price_panel.shape)

https://api.quiverquant.com/beta/bulk/congresstrading
45575 trades | 120 tickers | 2022-01-01 to 2026-07-01
price panel: (1126, 118)


In [5]:
rebalance_dates = pd.bdate_range(start, end, freq="W-FRI")
score_fn = lambda d: signal.compute(as_of=d, trades=trades)["score"]

result = E.event_study(score_fn, price_panel, rebalance_dates, horizon=HORIZON, n_buckets=5)
s = result["summary"]
if s["n_periods"] == 0:
    print("[!] no periods had enough data — confirm the price panel populated above.")
else:
    print(f"periods tested: {s['n_periods']}")
    print(f"mean IC: {s['mean_ic']:+.3f}  (positive {s['ic_positive_share']:.0%} of periods)")
    print(f"mean long-short ({HORIZON}d): {s['mean_long_short']:+.3%}  "
          f"(positive {s['ls_positive_share']:.0%} of periods)")

periods tested: 230
mean IC: +0.018  (positive 54% of periods)
mean long-short (21d): +0.691%  (positive 57% of periods)


### Forward return by score bucket

If the score carries information, the bars rise from the lowest bucket to the
highest. A flat or jagged profile says the score is not separating winners from
losers over this window.


In [6]:
import plotly.express as px

bm = result["bucket_means"]
if bm.empty:
    print("no bucket data to plot")
else:
    df = bm.reset_index(); df.columns = ["bucket", "mean_fwd_return"]
    df["bucket"] = df["bucket"].map({0:"1 (lowest)",1:"2",2:"3",3:"4",4:"5 (highest)"})
    fig = px.bar(df, x="bucket", y="mean_fwd_return",
                 title=f"Mean {HORIZON}-day forward return by congress-score bucket",
                 labels={"mean_fwd_return": "mean forward return"})
    fig.update_traces(marker_color="#2563eb")
    fig.update_layout(height=380, yaxis_tickformat=".1%")
    fig.show()

### Strategy view — separating alpha from beta

Three curves on a non-overlapping monthly schedule. **Long-only top bucket** holds the
top-scoring names; its returns mix the signal with full market exposure. **Universe**
equal-weights every name, a buy-and-hold proxy for that market exposure. **Long-short**
holds the top bucket and shorts the bottom, which cancels most of the market and leaves
the signal's own contribution. Comparing the three shows how much of the long-only
result is the score versus the market it rode.


In [7]:
import plotly.graph_objects as go

eq_top, rets_top = E.long_top_curve(score_fn, price_panel, rebalance_dates, horizon=HORIZON, top_frac=0.2)
eq_ls,  rets_ls  = E.long_short_curve(score_fn, price_panel, rebalance_dates, horizon=HORIZON, top_frac=0.2)
eq_bm,  _        = E.benchmark_curve(price_panel, rebalance_dates, horizon=HORIZON)

m_top, m_ls = E.metrics(rets_top, HORIZON), E.metrics(rets_ls, HORIZON)
if m_top["n_trades"] == 0:
    print("[!] no completed trades — the backtest produced no events (see the price-panel note above).")
else:
    print(f"long-only top : CAGR {m_top['cagr']:+.1%} | Sharpe {m_top['sharpe']:.2f} "
          f"| maxDD {m_top['max_drawdown']:.1%} | hit {m_top['hit_rate']:.0%} | n {m_top['n_trades']}")
    print(f"long-short    : CAGR {m_ls['cagr']:+.1%} | Sharpe {m_ls['sharpe']:.2f} "
          f"| maxDD {m_ls['max_drawdown']:.1%} | hit {m_ls['hit_rate']:.0%} | n {m_ls['n_trades']}")
    fig = go.Figure()
    fig.add_scatter(y=eq_top.values, name="Top bucket (long-only)", line=dict(color="#16a34a"))
    fig.add_scatter(y=eq_bm.values,  name="Universe (equal-weight)", line=dict(color="#9ca3af", dash="dash"))
    fig.add_scatter(y=eq_ls.values,  name="Long-short (market-neutral)", line=dict(color="#2563eb"))
    fig.update_layout(title="Equity curves (growth of 1)", height=400,
                      xaxis_title="period", yaxis_title="equity")
    fig.show()

long-only top : CAGR +9.1% | Sharpe 0.46 | maxDD -30.7% | hit 57% | n 46
long-short    : CAGR +12.9% | Sharpe 0.80 | maxDD -13.4% | hit 61% | n 46


### Feature attribution — does committee alignment pay?

This is how a feature earns its weight. We split every scored name by whether its
buying was committee-aligned and compare forward returns. A positive gap argues for
weight; a gap near zero argues for trimming it.


In [8]:
panel_fn = lambda d: signal.compute(as_of=d, trades=trades)
attr = E.feature_buckets(panel_fn, price_panel, rebalance_dates,
                         feature="committee_alignment", horizon=HORIZON)
if attr["with_n"] == 0 and attr["without_n"] == 0:
    print("no events available for feature attribution")
else:
    print(f"committee-aligned buys : mean {HORIZON}d return {attr['with_feature_mean']:+.3%}  "
          f"(n={attr['with_n']})")
    print(f"non-aligned buys       : mean {HORIZON}d return {attr['without_feature_mean']:+.3%}  "
          f"(n={attr['without_n']})")

committee-aligned buys : mean 21d return +2.205%  (n=1029)
non-aligned buys       : mean 21d return +1.195%  (n=14931)


## Reading the results honestly

- **The IC and the long-short consistency matter more than any single number.** A
  modest IC (say $0.03$–$0.06$) that is positive in a clear majority of periods is a
  real, tradeable signal. A large figure from a handful of periods is usually noise.
- **Read the long-short, not the long-only, for the signal's own edge.** The long-only
  top bucket carries full market exposure, so its Sharpe and drawdown mostly describe
  the market it rode. The long-short curve cancels that beta and shows what the score
  itself contributed.
- **Costs and regime still apply.** The curves exclude transaction costs and assume
  clean fills, and an edge over this window may not persist; the honest output is a
  confidence range with caveats.
- **Coverage shapes the committee feature.** With static sectors, alignment fires for
  the mapped large caps; enabling the yfinance fallback (`LiveEnrichment(use_yf_sectors=True)`)
  broadens it at the cost of API calls.

This backtest is also what turns the feature weights from priors into evidence: the
bucket profile and feature attribution show which features to up-weight and which to trim.


## Recap and next

The project now measures, not just ranks. The event-study engine reports whether the
congress score precedes returns, with an information coefficient, a bucket profile, a
strategy equity curve, and feature attribution that calibrates the weights — validated
against synthetic data where the answer is known.

Next, the remaining signals (government contracts, lobbying, off-exchange) join on the
same `BaseSignal` contract, the composite blends them, and the play classifier maps a
composite score and volatility conditions to a trade structure. The dashboard in
notebook `03` surfaces today's candidates with the historical edge behind each.
